# FlyFFN v2 vs SmolLM2-135M — progressive FFN-only replacement

Standard SmolLM2 attention stays unchanged. Non-anchor FFNs start **exactly dense**, then attempt the progressive schedule **8→6→4→3→2 shards**. Every stage is accepted only when the full-model CE gap stays inside the configured quality gate; otherwise that group is rolled back. Every fourth FFN remains a dense anchor.

This is a quality prototype: dense/sparse blending still computes all shards. A fused selected-shard kernel is a later speed optimization.

In [ ]:
#@title 1. Update repository, install, and syntax-check
import pathlib, subprocess, sys
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),
                'transformers>=4.56','datasets>=3.0','accelerate>=1.0','huggingface_hub>=0.34',
                'pandas>=2.0','requests>=2.31','tqdm>=4.66'], check=True)
for p in [REPO_DIR/'src'/'tinycenn_lm'/'smollm2_flyffn_v2.py', REPO_DIR/'scripts'/'run_smollm2_flyffn_v2.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(p)], check=True)
print('✓ FlyFFN-v2 module and runner syntax OK')
print('Ready:', REPO_DIR)


In [ ]:
#@title 2. Configuration
RUN_MODE = 'quick' #@param ['quick','strong']
SEQ_LEN = 128 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
FLY_NODES = 256 #@param {type:'integer'}
ROUTER_RANK = 64 #@param {type:'integer'}
MAX_EDGES = 2048 #@param {type:'integer'}
NUM_SHARDS = 8 #@param {type:'integer'}
GRAPH_STEPS = 1 #@param {type:'integer'}
GRAPH_MIX_INIT = 0.50 #@param {type:'number'}
ANCHOR_EVERY = 4 #@param {type:'integer'}
MAX_CE_GAP = 0.45 #@param {type:'number'}
RUN_REWIRED_CONTROL = True #@param {type:'boolean'}
OUTPUT_DIR = REPO_DIR/'results'/'flyffn_v2_smollm2_135m'
print('Attention: standard SmolLM2 (unchanged)')
print('Schedule: 8→6→4→3→2 | dense anchors every', ANCHOR_EVERY, 'layers | CE gate <=', MAX_CE_GAP)


In [ ]:
#@title 3. Train / evaluate — live output
import os, subprocess, sys
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_smollm2_flyffn_v2.py'),
     '--run-mode',RUN_MODE,'--seq-len',str(SEQ_LEN),'--batch-size',str(BATCH_SIZE),
     '--fly-nodes',str(FLY_NODES),'--router-rank',str(ROUTER_RANK),'--max-edges',str(MAX_EDGES),
     '--num-shards',str(NUM_SHARDS),'--graph-steps',str(GRAPH_STEPS),'--graph-mix-init',str(GRAPH_MIX_INIT),
     '--anchor-every',str(ANCHOR_EVERY),'--max-ce-gap',str(MAX_CE_GAP),'--output-dir',str(OUTPUT_DIR)]
if RUN_REWIRED_CONTROL: cmd.append('--rewired')
print('='*92); print('FlyFFN-v2 progressive FFN experiment'); print('Command:', ' '.join(cmd)); print('='*92)
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['TQDM_MININTERVAL']='1'
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
for line in iter(p.stdout.readline,''):
    print(line, end='', flush=True)
rc=p.wait(); print('\nFinished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Results
import json, pandas as pd
from IPython.display import display
summary=pd.read_csv(OUTPUT_DIR/'summary.csv',index_col=0)
report=json.loads((OUTPUT_DIR/'report.json').read_text())
samples=json.loads((OUTPUT_DIR/'samples.json').read_text())
display(summary)
print('\nCHECKS')
print('Architecture:',report['architecture'])
print('Attention unchanged:',report['attention_unchanged'])
print('FlyFFN-v2 layers:',report['flyffn_layers'])
print('Dense FFN anchors:',report['dense_anchor_layers'])
print('Dense-equivalence max logit diff:',report['dense_equivalence_biological_max_abs_logit_diff'])
print('Device:',report['device'],'| dtype:',report['dtype'])
print('\nKEY METRICS')
for k in ['fly_ce_gap_vs_smollm2','fly_ppl_ratio_vs_smollm2','parameter_ratio_fly_over_smollm2','decode_speed_ratio_fly_over_smollm2','biological_topology_ce_gain','biological_topology_ppl_gain_pct']:
    if k in report: print(k,':',report[k])
print('\nFINAL BIOLOGICAL ROUTING SCHEDULE')
for layer,state in report['biological_routing_schedule'].items(): print(f'layer {layer:>2}: k={state["active_k"]}, mix={state["route_mix"]:.2f}')
print('\nSAMPLES')
for x in samples:
    print('='*90); print('PROMPT:',x['prompt']); print(x['text'])
